# (1)

In [217]:
import pandas as pd
##WIP: add seaborn and matplotlib here.

#Do not want to tamper on the base data. 
#We'll save the changes in a new .csv file as one of the final steps.
birdstrikes_raw = pd.read_csv('../birdstrikes.csv', parse_dates=['Flight Date'])
birdstrikes = birdstrikes_raw.copy()

#Table size. (Should be 10000, 14)
print(f'Size/shape: {birdstrikes.shape}')

#Columns.
print(f'Columns: {list(birdstrikes.columns.values)}')

#General idea of the actual values we'll be dealing with.
print(birdstrikes.value_counts().head(10))

#These first 10 entries have 0 on three columns, being: Cost Other, Cost Repair, and Cost Total.
#Therefore, we'll conduct a quick check if this applies for all other entries as well.
print(len(birdstrikes[birdstrikes['Cost Other'] == 0])) #Count: 9928
print(len(birdstrikes[birdstrikes['Cost Repair'] == 0])) #Count: 9822
print(len(birdstrikes[birdstrikes['Cost Total $'] == 0])) #Count: 9791

Size/shape: (10000, 14)
Columns: ['Airport Name', 'Aircraft Make Model', 'Effect Amount of damage', 'Flight Date', 'Aircraft Airline Operator', 'Origin State', 'Phase of flight', 'Wildlife Size', 'Wildlife Species', 'Time of day', 'Cost Other', 'Cost Repair', 'Cost Total $', 'Speed IAS in knots']
Airport Name                    Aircraft Make Model  Effect Amount of damage  Flight Date  Aircraft Airline Operator  Origin State  Phase of flight  Wildlife Size  Wildlife Species       Time of day  Cost Other  Cost Repair  Cost Total $  Speed IAS in knots
NEW ORLEANS INTL                B-737-300            Substantial              1990-01-11   SOUTHWEST AIRLINES         Louisiana     Take-off run     Small          Rock pigeon            Day          0           0            0             140.0                 1
WASHINGTON DULLES INTL ARPT     BA-146               Medium                   1990-03-03   AIR WISCONSIN AIRLINES     DC            Climb            Large          Unknown bird - la

Seen in its entirety, a row describes the circumstances of the flight through a handful of different day:
What type of flight from where, and when the incident happened both in terms of day/night and flight phase.

Most values do not have any specified costs, between about 0.5-2.0% of the entries have anything beyond 0.
The zeroes do not have to mean a cost was never inserted, but actually that no cost to fix the plane in any way was needed.
Since there are birds, it is highly unlikely that any of them would weigh more than a human, and
collision between a human and a plane would highly likely and realistically only cause harm to the human in most likely.

We'll keep these values; they could be outliers of further interest.

# (2)

In [218]:
#To know if something could be incorrect, it can be smart to first get what datatypes there are.
birdstrikes.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Airport Name               10000 non-null  str           
 1   Aircraft Make Model        10000 non-null  str           
 2   Effect Amount of damage    1061 non-null   str           
 3   Flight Date                10000 non-null  datetime64[us]
 4   Aircraft Airline Operator  10000 non-null  str           
 5   Origin State               10000 non-null  str           
 6   Phase of flight            10000 non-null  str           
 7   Wildlife Size              10000 non-null  str           
 8   Wildlife Species           10000 non-null  str           
 9   Time of day                10000 non-null  str           
 10  Cost Other                 10000 non-null  int64         
 11  Cost Repair                10000 non-null  int64         
 12  Cost Total $    

**Speed IAS in knots** has 7164 non-null entries. 
**Effect amount of damage** has 1061 non-null entries; worth investigating.

Another point of curiosity is why only one of the Cost columns have a $ in it: **Cost Total \$**. 

In [219]:
#How many entries that are duplicates in the eyes of each other, 
#and how many duplicates there are (ignoring the first entry found).
print(birdstrikes.duplicated(keep=False).sum())
print(birdstrikes.duplicated().sum())

birdstrikes[birdstrikes.duplicated()]

45
24


,Airport Name,Aircraft Make Model,Effect Amount of damage,Flight Date,Aircraft Airline Operator,Origin State,Phase of flight,Wildlife Size,Wildlife Species,Time of day,Cost Other,Cost Repair,Cost Total $,Speed IAS in knots
341,ATLANTIC CITY INTL,F-16,NaN,1990-10-15,MILITARY,New Jersey,Landing Roll,Medium,Unknown bird or bat,Night,0,0,0,0.0
1114,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1992-04-14,MILITARY,Louisiana,Approach,Medium,Unknown bird or bat,Night,0,0,0,150.0
1132,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1992-04-23,MILITARY,Louisiana,Approach,Medium,Unknown bird or bat,Night,0,0,0,140.0
1268,GREATER PITTSBURGH,DC-9-30,NaN,1992-06-15,US AIRWAYS*,Pennsylvania,Climb,Small,Unknown bird - small,Day,0,0,0,150.0
1814,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1993-04-28,MILITARY,Louisiana,Approach,Medium,Unknown bird or bat,Night,0,0,0,180.0
2081,CHARLESTON AFB/INTL ARPT,C-141B,NaN,1993-08-18,MILITARY,South Carolina,Climb,Medium,Unknown bird or bat,Day,0,0,0,130.0
2897,CHARLESTON AFB/INTL ARPT,C-141B,NaN,1994-10-10,MILITARY,South Carolina,Approach,Medium,Unknown bird or bat,Night,0,0,0,160.0
3043,HONOLULU INTL ARPT,C-141,NaN,1995-01-10,MILITARY,Hawaii,Approach,Medium,Unknown bird - medium,Day,0,0,0,128.0
6190,CHARLESTON AFB/INTL ARPT,C-17A,NaN,1998-11-04,MILITARY,South Carolina,Approach,Medium,Unknown bird or bat,Night,0,0,0,230.0
6236,CHARLESTON AFB/INTL ARPT,C-17A,NaN,1998-12-04,MILITARY,South Carolina,Approach,Medium,Unknown bird or bat,Day,0,0,0,130.0


The duplicates that are identical on all columns will not give any new insights to us.
It is way too unlikely that entries matching on everything, from model, airport AND collision time, would happen.
We'll start off by deleting these.

In [220]:
#keep is 'first' by default, so we do not specify anything.
birdstrikes = birdstrikes.drop_duplicates(keep='first')

#Count as a safety measure.
print(birdstrikes.duplicated().sum())

0


Most birdstrikes have reoccurring aircraft models, based on the fact that this gives 9751 Trues.

In [221]:
birdstrikes.duplicated(subset = 'Aircraft Make Model').value_counts()

True     9751
False     225
Name: count, dtype: int64

A sample used as a check - that does not however answer about the entire dataset - shows that entries with the same models do not 
have too identical values over all the columns.

In [222]:
birdstrikes[birdstrikes.duplicated(subset = 'Aircraft Make Model')].head(20)

,Airport Name,Aircraft Make Model,Effect Amount of damage,Flight Date,Aircraft Airline Operator,Origin State,Phase of flight,Wildlife Size,Wildlife Species,Time of day,Cost Other,Cost Repair,Cost Total $,Speed IAS in knots
4,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1990-01-12,MILITARY,Louisiana,Climb,Medium,Unknown bird or bat,Day,0,0,0,160.0
5,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1990-02-02,MILITARY,Louisiana,Take-off run,Medium,Unknown bird or bat,Day,0,0,0,180.0
6,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1990-02-06,MILITARY,Louisiana,Climb,Medium,Unknown bird or bat,Dusk,0,0,0,140.0
8,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1990-03-01,MILITARY,Louisiana,Climb,Medium,Unknown bird or bat,Day,0,0,0,180.0
9,BARKSDALE AIR FORCE BASE ARPT,B-52,NaN,1990-03-02,MILITARY,Louisiana,Take-off run,Medium,Unknown bird or bat,Dusk,0,0,0,130.0
11,BARKSDALE AIR FORCE BASE ARPT,B-52,NaN,1990-03-08,MILITARY,Louisiana,Approach,Medium,Unknown bird or bat,Night,0,0,0,150.0
13,CHARLESTON AFB/INTL ARPT,C-141B,NaN,1990-03-09,MILITARY,South Carolina,Approach,Medium,Unknown bird or bat,Night,0,0,0,190.0
14,BARKSDALE AIR FORCE BASE ARPT,B-52,NaN,1990-03-09,MILITARY,Louisiana,Take-off run,Medium,Unknown bird or bat,Night,0,0,0,150.0
16,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1990-03-19,MILITARY,Louisiana,Approach,Medium,Unknown bird or bat,Dusk,0,0,0,150.0
18,BARKSDALE AIR FORCE BASE ARPT,KC-10A,NaN,1990-04-04,MILITARY,Louisiana,Approach,Medium,Unknown bird or bat,Night,0,0,0,135.0


In [223]:
birdstrikes[birdstrikes['Aircraft Airline Operator'] == 'MILITARY']['Cost Total'].agg(['mean', 'median'])

KeyError: 'Cost Total'

In [ ]:
#The few values raise the mean a LOT.
birdstrikes[
    (birdstrikes['Aircraft Airline Operator'] == 'MILITARY') &
    (~birdstrikes['Effect Amount of damage'].isin(['B', 'C']))
]['Cost Total'].agg(['mean', 'median'])

mean      186.006258
median      0.000000
Name: Cost Total, dtype: float64

To check if there could be any correlation between the Cost columns, before we change the name of the Cost Final $ column,
we'll check different values with an aggregation.

In [ ]:
birdstrikes.groupby('Effect Amount of damage')[['Cost Other', 'Cost Repair', 'Cost Total $']].agg(['mean', 'median'])

KeyError: "Columns not found: 'Cost Total $'"

The means in Other and Repair are the sum of Total. Safe to say that they do relate to each other, and we can rename the column.

In [ ]:
birdstrikes = birdstrikes.rename(columns={'Cost Total $' : 'Cost Total'})

In [ ]:
birdstrikes.columns

Index(['Airport Name', 'Aircraft Make Model', 'Effect Amount of damage',
       'Flight Date', 'Aircraft Airline Operator', 'Origin State',
       'Phase of flight', 'Wildlife Size', 'Wildlife Species', 'Time of day',
       'Cost Other', 'Cost Repair', 'Cost Total', 'Speed IAS in knots'],
      dtype='str')

In [ ]:
birdstrikes[
    birdstrikes['Effect Amount of damage'].isin(['B', 'C'])
]

,Airport Name,Aircraft Make Model,Effect Amount of damage,Flight Date,Aircraft Airline Operator,Origin State,Phase of flight,Wildlife Size,Wildlife Species,Time of day,Cost Other,Cost Repair,Cost Total,Speed IAS in knots
299,ATLANTIC CITY INTL,C-137C,C,1990-09-26,MILITARY,New Jersey,Climb,Medium,Unknown bird - medium,Day,0,23656,23656,150.0
441,BARKSDALE AIR FORCE BASE ARPT,B-52G,C,1990-11-30,MILITARY,Louisiana,Climb,Small,Cattle egret,Day,0,67435,67435,175.0
598,BARKSDALE AIR FORCE BASE ARPT,B-52G,C,1991-05-14,MILITARY,Louisiana,Approach,Large,Turkey vulture,Day,0,160764,160764,140.0
648,CHARLESTON AFB/INTL ARPT,C-141B,C,1991-06-22,MILITARY,South Carolina,Approach,Medium,Unknown bird or bat,Day,0,37234,37234,200.0
1493,GREATER PITTSBURGH,KC-135E,C,1992-09-12,MILITARY,Pennsylvania,Approach,Medium,Unknown bird - medium,Day,0,117399,117399,200.0
2543,ATLANTIC CITY INTL,F-16,C,1994-05-19,MILITARY,New Jersey,Approach,Medium,Unknown bird - medium,Day,0,20439,20439,180.0
3381,CHARLESTON AFB/INTL ARPT,C-17A,C,1995-08-04,MILITARY,South Carolina,Approach,Medium,Unknown bird or bat,Day,0,69282,69282,230.0
5270,BARKSDALE AIR FORCE BASE ARPT,B-52H,C,1997-11-07,MILITARY,Louisiana,Approach,Medium,Unknown bird or bat,Dusk,0,21514,21514,155.0
5753,BARKSDALE AIR FORCE BASE ARPT,B-52H,C,1998-07-30,MILITARY,Louisiana,Approach,Large,Great blue heron,Night,0,21360,21360,150.0
7407,CHARLESTON AFB/INTL ARPT,C-17A,C,2000-04-24,MILITARY,South Carolina,Climb,Medium,Unknown bird or bat,Day,0,32073,32073,NaN


# (3)

Currently just scrap, but it will lay the basis for my decision whether something will stay or not.

**Speed IAS in knots** has 7164 non-null entries. 
**Effect amount of damage** has 1061 non-null entries; worth investigating.

In [ ]:
birdstrikes['Effect Amount of damage'].value_counts(dropna=False)

Effect Amount of damage
NaN            8916
Minor           548
Substantial     311
Medium          186
C                14
B                 1
Name: count, dtype: int64

Most values for the categories we do understand have 0 as a value. 
So it would be smarter to actually look at the scenario when these are not 0.
Since we cannot say exactly what B and C represent, and since they are so few, there may be a possibility that we actually remove these later on.

In [ ]:
birdstrikes.groupby('Effect Amount of damage',
)['Cost Total $'].agg(['count', 'mean', 'median'])

KeyError: 'Column not found: Cost Total $'

In [ ]:
#We can check what the count, mean and median are on these, and see if they relate to each other or not.
birdstrikes[birdstrikes['Cost Total $'] != 0][['Cost Total $', 'Cost Repair', 'Cost Other']]
#If this is true, we can rename the column Cost Total $ so that it does not have that $, because these columns must look at the same thing, right? 

,Cost Total $,Cost Repair,Cost Other
15,4175,4175,0
107,48704,0,48704
276,5218,5218,0
299,23656,23656,0
388,136109,136109,0
...,...,...,...
9519,24087,21552,2535
9578,317,317,0
9881,636405,636405,0
9924,397639,27459,370180


# (4)

Merge WIP. 
Likely a left merge, though I am not sure yet.